# Grid Search Demo (All 3 Models)

This notebook runs hyperparameter grid search for ItemKNN, SVD, and LightFM using the shared train/validation split.

In [1]:
from pathlib import Path

import pandas as pd

from src.evaluation.grid_search import GridSearchConfig
from src.evaluation.grid_search import RecommenderGridSearch

/home/samuel/Documents/data-mining-recommender-systems/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Resolve project root from notebook folder.
project_root_path = Path.cwd()
if not (project_root_path / "src").exists():
    project_root_path = project_root_path.parent

train_ratings_path = project_root_path / "data" / "processed" / "notebook_demo" / "ratings_train_split.csv"
validation_ratings_path = project_root_path / "data" / "processed" / "notebook_demo" / "ratings_validation_split.csv"
movies_features_path = project_root_path / "data" / "processed" / "movies_cleaned.csv"
grid_output_directory_path = project_root_path / "data" / "processed" / "grid_search" / "notebook_run"

train_dataframe = pd.read_csv(train_ratings_path)
validation_dataframe = pd.read_csv(validation_ratings_path)
movies_dataframe = pd.read_csv(movies_features_path)

print(f"Train rows: {len(train_dataframe)}")
print(f"Validation rows: {len(validation_dataframe)}")
print(f"Movies rows: {len(movies_dataframe)}")

Train rows: 95251
Validation rows: 2550
Movies rows: 9742


In [3]:
# Configure search for all three models.
# Set maximum_trials_per_model=None for full exhaustive runs.
search_config = GridSearchConfig(
    selected_model_names=["itemknn", "svd", "lightfm"],
    metric_name="ndcg_at_k",
    number_of_recommendations=10,
    relevance_threshold=4.0,
    maximum_trials_per_model=3,
    output_directory_path=grid_output_directory_path,
)
search_runner = RecommenderGridSearch(search_config=search_config)

In [4]:
# Run grid search and collect best trial per model.
model_results = search_runner.run(
    train_dataframe=train_dataframe,
    validation_dataframe=validation_dataframe,
    movies_dataframe=movies_dataframe,
)

print(f"Saved artifacts in: {grid_output_directory_path}")
for model_result in model_results:
    metric_value = getattr(model_result.best_trial.evaluation_result, model_result.metric_name)
    print(
        f"Model={model_result.model_name}, "
        f"best_trial={model_result.best_trial.trial_index}, "
        f"{model_result.metric_name}={metric_value:.4f}, "
        f"params={model_result.best_trial.parameter_values}"
    )

In [8]:
# Load saved best-result files for quick review.
best_rows = []
for model_name in ["itemknn", "svd", "lightfm"]:
    best_json_path = grid_output_directory_path / model_name / "best_result.json"
    if not best_json_path.exists():
        continue
    best_payload = pd.read_json(best_json_path, typ="series")
    best_rows.append(best_payload)


if best_rows:
     results = pd.DataFrame(best_rows)
     # Extract NDCG into a top-level column for easier sorting and filtering.
     results["ndcg_at_k"] = results["best_metrics"].apply(
         lambda metrics_payload: metrics_payload.get("ndcg_at_k") if isinstance(metrics_payload, dict) else None
     )
else:
    results = None
    print("No best_result.json files found yet.")

results

,model_name,metric_name,best_trial_index,best_parameters,best_metrics,ndcg_at_k
0,itemknn,ndcg_at_k,3,"{'number_of_neighbors': 20, 'minimum_neighbors...","{'rmse_value': 0.89571491088605, 'mae_value': ...",0.631075
1,svd,ndcg_at_k,1,"{'number_of_factors': 20, 'number_of_epochs': ...","{'rmse_value': 0.8406043230338701, 'mae_value'...",0.683569
2,lightfm,ndcg_at_k,2,"{'number_of_components': 16, 'number_of_epochs...","{'rmse_value': 5.233517396986205, 'mae_value':...",0.595367


In [6]:
# Show all-trials table for one model as an example.
svd_trials_path = grid_output_directory_path / "svd" / "all_trials.csv"
if svd_trials_path.exists():
    pd.read_csv(svd_trials_path).head()
else:
    print(f"Missing file: {svd_trials_path}")